In [ ]:
# Cell 1: Import all required libraries

import os                                    # build cross-platform file paths from parts
import datetime                              # strip timezone info from EDF header timestamps
import mne                                   # read EDF files and access EEG channel metadata
import numpy as np                           # fast numerical array operations and statistics
import pandas as pd                          # read Excel annotation files and parse datetime columns
from scipy.signal import iirnotch, filtfilt  # design IIR notch filter and apply zero-phase filtering

## EEG Preprocessing Pipeline

This notebook preprocesses raw EEG recordings stored as `.edf` files before they are fed into a deep learning model.

Each recording goes through **three sequential steps**:

| Step | What it does | Why |
|------|-------------|-----|
| **1. 50 Hz notch filter** | Removes powerline interference using a narrow IIR notch filter applied zero-phase (forward + backward pass). | European mains hum at 50 Hz contaminates almost every EEG recording and must be removed before frequency-domain analysis or model training. |
| **2. Saturation / clipping removal** | Samples that hit the amplifier voltage rail (hard clip) or stay constant for >= 5 consecutive samples (flat-line dropout) are marked as NaN and linearly interpolated. | Saturated samples contain no neural information and would corrupt normalisation statistics and model weights. |
| **3. Robust z-score (MAD)** | Each channel is standardised using the median and Median Absolute Deviation (MAD) rather than mean +/- std. | The median and MAD are resistant to large spike artefacts that may remain after clipping removal, making normalisation stable across sessions. |

The function `clean_and_normalise_signal()` in the next cell implements this pipeline and returns a **1-D NumPy array** for a single selected EEG channel.

In [ ]:
# ── Cell 3: clean_and_normalise_signal() — full EEG preprocessing function ────────────────

def clean_and_normalise_signal(
    edf_path,           # full path to the .edf file (e.g. '/data/edfs/subject01.edf')
    channel_name,       # exact channel label as stored in the EDF header (e.g. 'EEG Fp1')
    notch_freq=50.0,    # powerline frequency to suppress in Hz — 50 Hz is the European standard
    notch_quality=30.0, # Q-factor of the notch filter: higher value = narrower, sharper notch
    sat_threshold=None, # hard clip level in raw signal units; auto-estimated from data if None
    flat_run_min=5,     # number of consecutive identical samples that flags a flat-line segment
    verbose=True        # print step-by-step progress messages when True
):
    """
    Load one EDF file, extract a single EEG channel, and return a clean,
    normalised 1-D NumPy array ready for model input.

    Parameters
    ----------
    edf_path      : str          Full path to the .edf file.
    channel_name  : str          Channel label as stored in the EDF header.
    notch_freq    : float        Frequency to remove (default 50 Hz).
    notch_quality : float        Notch Q-factor controlling width (default 30).
    sat_threshold : float|None   Clip level in raw signal units; auto if None.
    flat_run_min  : int          Consecutive equal samples before a run is flagged.
    verbose       : bool         Print progress when True.

    Returns
    -------
    signal_norm : np.ndarray, shape (n_samples,), dtype float32
        Preprocessed, normalised EEG signal for the chosen channel.
    """

    # ── Step 1: Load the EDF file with MNE ────────────────────────────────────
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)  # preload=True reads all data into RAM
    sfreq = raw.info['sfreq']                    # sampling frequency in Hz stored in the EDF header

    if verbose:
        print(f'[1] Loaded   : {edf_path}')      # confirm which file was opened
        print(f'    Channels : {len(raw.ch_names)}  |  Fs: {sfreq} Hz  |  Duration: {raw.times[-1]:.1f} s')
        print(f'    Available channels: {raw.ch_names}')  # list all channels so the caller can verify

    # ── Step 2: Extract the requested single EEG channel ──────────────────────
    if channel_name not in raw.ch_names:         # guard: raise a clear error if the channel does not exist
        raise ValueError(
            f"Channel '{channel_name}' not found. "
            f"Available channels: {raw.ch_names}"
        )

    raw_ch = raw.copy().pick_channels([channel_name])          # keep only the requested channel
    signal = raw_ch.get_data().squeeze().astype(np.float64)    # shape (n_samples,) — squeeze removes the channel axis

    if verbose:
        print(f"[2] Extracted channel '{channel_name}'  |  {len(signal)} samples")  # confirm extraction

    # ── Step 3: 50 Hz notch filter — remove powerline interference ─────────────
    b, a = iirnotch(
        w0=notch_freq / (sfreq / 2),  # normalise notch frequency to the Nyquist frequency (must be in [0, 1])
        Q=notch_quality               # Q-factor: Q=30 gives a notch ~1.7 Hz wide at 50 Hz
    )
    signal = filtfilt(b, a, signal)   # zero-phase filtering: forward then backward pass cancels phase distortion

    if verbose:
        print(f'[3] Notch filter applied at {notch_freq} Hz  (Q={notch_quality})')  # confirm filter step

    # ── Step 4a: Hard-clip saturation detection ────────────────────────────────
    if sat_threshold is None:                    # auto-estimate threshold when not provided by caller
        sat_threshold = np.percentile(np.abs(signal), 99.9)  # 99.9th percentile catches rail-to-rail clipping

    signal = signal.astype(np.float64)           # ensure float64 so we can assign NaN
    signal[np.abs(signal) >= sat_threshold] = np.nan  # replace every saturated sample with NaN

    if verbose:
        n_sat = int(np.sum(np.isnan(signal)))    # count how many samples were clipped
        print(f'[4a] Saturation threshold: {sat_threshold:.4e}  |  {n_sat} samples removed ({100*n_sat/len(signal):.2f}%)')

    # ── Step 4b: Flat-line (soft-clip / dropout) detection ────────────────────
    equal = np.concatenate(([False], np.diff(signal) == 0, [False]))  # True where adjacent samples are identical
    run_starts = np.where(~equal[:-1] &  equal[1:])[0]               # index where each flat run begins
    run_ends   = np.where( equal[:-1] & ~equal[1:])[0] + 1           # index just after each flat run ends

    n_flat = 0                                   # accumulate total flat samples removed for reporting
    for s, e in zip(run_starts, run_ends):       # iterate over every detected flat-line segment
        if (e - s) >= flat_run_min:              # only flag runs long enough to be real artefacts
            signal[s:e] = np.nan                 # mark the entire flat segment as missing
            n_flat += (e - s)                    # add segment length to the running count

    if verbose:
        print(f'[4b] Flat-line segments removed: {n_flat} samples')  # report flat-line removal

    # ── Step 4c: Linear interpolation over all NaN gaps ───────────────────────
    nan_mask = np.isnan(signal)                  # Boolean mask: True wherever a sample was removed
    good_idx = np.where(~nan_mask)[0]            # indices of all remaining valid (non-NaN) samples

    if len(good_idx) > 1:                        # need at least 2 good points to interpolate
        signal = np.interp(
            np.arange(len(signal)),              # query every sample index
            good_idx,                            # x-coordinates of known-good samples
            signal[good_idx]                     # y-values of known-good samples
        )
    elif len(good_idx) == 0:                     # channel is completely bad — cannot recover
        signal[:] = 0.0                          # zero the channel rather than leaving NaN downstream
        if verbose:
            print(f"    Warning: channel '{channel_name}' had no valid samples and was zeroed.")

    if verbose:
        print(f'    Interpolation done  |  NaN remaining: {int(np.sum(np.isnan(signal)))}')  # confirm no NaN left

    # ── Step 5: Robust z-score normalisation (MAD) ────────────────────────────
    # Formula:  z = (x - median(x)) / (1.4826 x MAD(x))
    # The 1.4826 factor makes MAD equal to std for Gaussian data.
    med = np.median(signal)                      # robust location estimate — not affected by residual spikes
    mad = np.median(np.abs(signal - med))        # Median Absolute Deviation — robust scale estimate
    mad_safe = mad if mad != 0.0 else 1.0        # guard against dead/constant channels where MAD == 0
    signal_norm = (signal - med) / (1.4826 * mad_safe)  # apply the robust z-score formula element-wise

    if verbose:
        print(f'[5] Robust z-score applied  |  median={med:.4e}  MAD={mad:.4e}')
        print(f'    Output shape : {signal_norm.shape}  |  range [{signal_norm.min():.2f}, {signal_norm.max():.2f}]')

    return signal_norm.astype(np.float32)        # cast to float32 to halve memory — sufficient for deep learning


In [ ]:
# ── Cell 5: process_batch() — batch process a folder of EDF files ──────────

def process_batch(
    data_dir,           # path to the folder containing all EDF files
    channel_name,       # EEG channel label to extract from every file (must be consistent across files)
    out_dir=None,       # folder where .npy files are saved; defaults to data_dir if not specified
    notch_freq=50.0,    # powerline frequency to suppress in Hz — passed through to clean_and_normalise_signal()
    notch_quality=30.0, # Q-factor of the notch filter — passed through to clean_and_normalise_signal()
    sat_threshold=None, # clipping threshold — passed through to clean_and_normalise_signal(); None = auto per file
    flat_run_min=5,     # minimum flat-line run length in samples — passed through to clean_and_normalise_signal()
    verbose=True        # print per-file progress when True
):
    """
    Batch-preprocess every EDF file in data_dir.

    Constructs the full file path for each EDF and passes it directly to
    clean_and_normalise_signal(), which only needs a path — not separate folder/name parts.
    Each output (a 1-D float32 NumPy array) is saved as a separate .npy file.
    Signals are NOT stacked because recordings may have different lengths.

    Parameters
    ----------
    data_dir      : str          Folder containing the EDF files.
    channel_name  : str          EEG channel label to extract from every file.
    out_dir       : str|None     Destination folder for .npy files (default: data_dir).
    notch_freq    : float        Notch frequency in Hz (default 50).
    notch_quality : float        Notch Q-factor (default 30).
    sat_threshold : float|None   Clipping threshold; auto-detected per file if None.
    flat_run_min  : int          Minimum flat-line run length to flag (default 5).
    verbose       : bool         Print progress per file when True.

    Returns
    -------
    summary : dict
        Keys are EDF file names; values are dicts with keys:
            'npy_path'  - absolute path of the saved .npy file, or
            'error'     - error message string if the file failed.
    """

    # ── Resolve output directory ──────────────────────────────────────────────
    if out_dir is None:                              # use data_dir as output location if not specified
        out_dir = data_dir
    os.makedirs(out_dir, exist_ok=True)             # create the output folder if it does not already exist

    # ── Collect all EDF files in the folder ───────────────────────────────────
    all_files = sorted(                             # sort for reproducible processing order
        f for f in os.listdir(data_dir)             # list every entry in the directory
        if f.lower().endswith('.edf')               # keep only files with an .edf extension
    )

    if len(all_files) == 0:                         # stop early if the folder contains no EDF files
        raise FileNotFoundError(f"No .edf files found in: {data_dir}")

    if verbose:
        print(f"Found {len(all_files)} EDF file(s) in: {data_dir}")
        print(f"Output folder : {out_dir}\n")

    summary = {}                                    # dict to collect per-file outcomes for the caller

    # ── Process each file ─────────────────────────────────────────────────────
    for idx, filename in enumerate(all_files):      # iterate over every EDF file in sorted order

        edf_path = os.path.join(data_dir, filename) # build the full path here, before calling clean_and_normalise_signal

        if verbose:
            print(f"[{idx+1}/{len(all_files)}] Processing: {filename}")

        try:
            signal = clean_and_normalise_signal(
                edf_path=edf_path,            # pass the complete path — clean_and_normalise_signal needs nothing else
                channel_name=channel_name,    # channel to extract — same label expected in every file
                notch_freq=notch_freq,        # 50 Hz notch frequency
                notch_quality=notch_quality,  # notch Q-factor
                sat_threshold=sat_threshold,  # clipping threshold (None = auto per file)
                flat_run_min=flat_run_min,    # minimum flat-line run length
                verbose=False                 # suppress per-step output during batch processing
            )

            # Build the output .npy file path: same stem as the EDF, with .npy extension.
            stem     = os.path.splitext(filename)[0]           # strip the .edf extension from the file name
            npy_name = stem + '.npy'                           # e.g. subject01_session1.npy
            npy_path = os.path.join(out_dir, npy_name)         # full path to the output file

            # Save the 1-D NumPy array to disk in NumPy's native binary format.
            np.save(npy_path, signal)                          # saves shape, dtype, and data — reload with np.load()

            summary[filename] = {'npy_path': npy_path}         # record success with the saved file path

            if verbose:
                print(f"    Saved -> {npy_path}  |  shape: {signal.shape}  dtype: {signal.dtype}")

        except Exception as exc:                               # catch any error so one bad file does not abort the batch
            summary[filename] = {'error': str(exc)}            # record the error message for this file
            print(f"    WARNING: {filename} failed — {exc}")   # always print failures regardless of verbose flag

    # ── Print final summary ───────────────────────────────────────────────────
    ok     = sum(1 for v in summary.values() if 'npy_path' in v)  # count successfully processed files
    failed = sum(1 for v in summary.values() if 'error'    in v)  # count failed files

    print(f"\nBatch complete: {ok} succeeded, {failed} failed out of {len(all_files)} files.")

    return summary  # return the per-file outcome dict so the caller can inspect or log it


In [ ]:
# Cell 5: Example usage of clean_and_normalise_signal() and process_batch()

# Single-file example
edf_path     = '/path/to/your/edf/folder/subject01_session1.edf'  # full path to a single EDF file
channel_name = 'EEG Fp1'                                          # channel label stored in the EDF header

clean_signal = clean_and_normalise_signal(  # run the full preprocessing pipeline on one file
    edf_path=edf_path,                      # full path — no need to split into folder + filename
    channel_name=channel_name,              # which EEG channel to extract and clean
    notch_freq=50.0,                        # suppress 50 Hz powerline interference
    notch_quality=30.0,                     # Q=30 keeps the notch narrow and precise
    sat_threshold=None,                     # None = auto-detect the clipping threshold from the data
    flat_run_min=5,                         # flag any run of 5 or more consecutive identical samples
    verbose=True                            # print each preprocessing step to the console
)

print(f'Output type  : {type(clean_signal)}')                                   # confirm np.ndarray
print(f'Output dtype : {clean_signal.dtype}')                                   # confirm float32
print(f'Output shape : {clean_signal.shape}')                                   # confirm 1-D (n_samples,)
print(f'Value range  : [{clean_signal.min():.2f}, {clean_signal.max():.2f}]')  # inspect z-score range

# Batch example: process all EDF files in a folder
data_dir = '/path/to/your/edf/folder'  # folder containing all EDF files
out_dir  = '/path/to/output/npy'       # folder where one .npy file per EDF will be written

summary = process_batch(            # process every .edf file found in data_dir
    data_dir=data_dir,              # scans this folder for .edf files
    channel_name=channel_name,      # same channel label extracted from every file
    out_dir=out_dir,                # saves one .npy per file here
    verbose=True                    # print per-file progress to the console
)

# Reload any saved signal from disk with a single call
saved_signal = np.load('/path/to/output/npy/subject01_session1.npy')  # returns the 1-D float32 array
print(f'Reloaded shape: {saved_signal.shape}')                         # confirm shape matches what was saved

## Segmentation and Labelling Pipeline

This section converts preprocessed EEG signals into labelled fixed-length segments ready for supervised deep learning.

### Annotation files
Seizure timing is stored in **Excel spreadsheets** (`.xlsx`), one per recording. Each spreadsheet has exactly two columns:

| Column | Format | Example |
|--------|--------|---------|
| `start_time` | `DD/MM/YYYY HH:MM:SS.fff` | `01/04/2023 14:32:10.000` |
| `end_time` | `DD/MM/YYYY HH:MM:SS.fff` | `01/04/2023 14:33:45.500` |

### Converting timestamps to sample indices
The EDF file header stores the recording start datetime (`raw.info['meas_date']`). Each annotation timestamp is converted to a sample index with:

```
sample_index = (timestamp - recording_start).total_seconds() * sampling_rate
```

### Sliding-window segmentation
Each labelled region is split into **5-second windows** with **2.5-second overlap** (50% overlap), producing segments of shape `(2500,)` at 500 Hz. Incomplete windows at the end of a region are discarded.

### Label classes and output folders
Each segment is assigned exactly one label based on its position relative to the seizure. Priority when regions overlap: **ictal > onset > offset > nonictal**.

| Label | Region | Saved to |
|-------|--------|----------|
| **Ictal** | Samples inside the seizure (start to end) | `output_dir/ictal/` |
| **Onset** | 30 s of samples immediately before each seizure start | `output_dir/onset/` |
| **Offset** | 30 s of samples immediately after each seizure end | `output_dir/offset/` |
| **Nonictal** | All remaining samples not covered by ictal, onset, or offset | `output_dir/nonictal/` |

### File naming convention
Each segment is saved as an individual `.npy` file:
```
{mouse_id}_{label}_{index:05d}.npy
# e.g.  m1_ictal_00012.npy  |  m1_nonictal_00204.npy
```
The `mouse_id` is extracted automatically from the EDF filename (without extension).

In [ ]:
# Cell 7: load_annotations() — read seizure timing from an Excel annotation file

def load_annotations(annotation_path):
    """
    Read a plain Excel file with two columns and return a DataFrame with
    both columns parsed as datetime objects.

    The Excel file must contain exactly two columns:
        start_time  –  seizure start, formatted as DD/MM/YYYY HH:MM:SS.fff
        end_time    –  seizure end,   formatted as DD/MM/YYYY HH:MM:SS.fff

    Parameters
    ----------
    annotation_path : str
        Full path to the .xlsx annotation file.

    Returns
    -------
    df : pd.DataFrame
        DataFrame with 'start_time' and 'end_time' columns as pandas Timestamps.
    """

    df = pd.read_excel(annotation_path)   # read the Excel file into a DataFrame; expects start_time and end_time columns

    # Parse both columns using the explicit format DD/MM/YYYY HH:MM:SS.fff.
    # %d = day, %m = month, %Y = 4-digit year, %H:%M:%S = time, %f = microseconds (covers milliseconds too).
    # dayfirst=False is irrelevant when an explicit format is given; the format string is unambiguous.
    dt_format = '%d/%m/%Y %H:%M:%S.%f'                                      # expected datetime format in the Excel file

    df['start_time'] = pd.to_datetime(df['start_time'], format=dt_format)   # parse start_time column to datetime
    df['end_time']   = pd.to_datetime(df['end_time'],   format=dt_format)   # parse end_time column to datetime

    return df   # return the DataFrame with both time columns as pandas Timestamp objects

In [ ]:
# Cell 8: get_recording_start_time() — extract the EDF recording start datetime

def get_recording_start_time(edf_path):
    """
    Load the EDF file header only and return the recording start time as a
    timezone-naive Python datetime object.

    Using preload=False avoids loading the full signal into RAM — only the
    header metadata is needed to retrieve meas_date.

    Parameters
    ----------
    edf_path : str
        Full path to the .edf file.

    Returns
    -------
    recording_start : datetime.datetime
        Timezone-naive recording start datetime extracted from the EDF header.
    """

    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)  # read header only — signal data is not loaded

    meas_date = raw.info['meas_date']                  # EDF header start datetime; may be timezone-aware (UTC)

    recording_start = meas_date.replace(tzinfo=None)   # strip timezone so arithmetic with annotation datetimes works cleanly

    return recording_start   # return timezone-naive datetime for use in compute_segment_labels()

In [ ]:
# ── compute_segment_labels() — convert annotation datetimes to sample indices ──

def compute_segment_labels(
    annotations_df,          # DataFrame returned by load_annotations() with start_time and end_time columns
    recording_start,         # timezone-naive datetime returned by get_recording_start_time()
    sampling_rate=500,       # samples per second of the preprocessed signal
    onset_buffer_sec=30,     # seconds of signal before each seizure start that form the onset region
    offset_buffer_sec=30     # seconds of signal after each seizure end that form the offset region
):
    """
    Convert seizure annotation datetimes to sample-level region boundaries.

    Priority rule when regions overlap: ictal > onset > offset.

    Parameters
    ----------
    annotations_df    : pd.DataFrame   Annotation table with 'start_time' and 'end_time'.
    recording_start   : datetime       Timezone-naive EDF recording start time.
    sampling_rate     : int            Samples per second (default 500).
    onset_buffer_sec  : float          Seconds before seizure start for onset region.
    offset_buffer_sec : float          Seconds after seizure end for offset region.

    Returns
    -------
    label_regions : list of dict
        One dict per seizure, each containing integer sample indices:
            ictal_start, ictal_end, onset_start, onset_end,
            offset_start, offset_end.
    """

    onset_samples  = int(onset_buffer_sec  * sampling_rate)   # convert onset buffer from seconds to number of samples
    offset_samples = int(offset_buffer_sec * sampling_rate)   # convert offset buffer from seconds to number of samples

    label_regions = []   # list to accumulate one region-dict per seizure

    for _, row in annotations_df.iterrows():   # iterate over each seizure annotation row

        # Convert absolute start/end datetimes to elapsed seconds from the recording start.
        start_sec = (row['start_time'].to_pydatetime() - recording_start).total_seconds()   # seconds from recording start to seizure start
        end_sec   = (row['end_time'].to_pydatetime()   - recording_start).total_seconds()   # seconds from recording start to seizure end

        ictal_start = int(start_sec * sampling_rate)   # first sample of the ictal (seizure) region
        ictal_end   = int(end_sec   * sampling_rate)   # last sample of the ictal region (exclusive upper bound)

        onset_start = max(0, ictal_start - onset_samples)   # onset begins onset_buffer_sec before the seizure; clamp to 0
        onset_end   = ictal_start                           # onset ends exactly where ictal begins (ictal takes priority)

        offset_start = ictal_end                                  # offset begins exactly where ictal ends
        offset_end   = ictal_end + offset_samples                 # offset extends offset_buffer_sec after the seizure

        # Build the region dict; ictal > onset > offset priority is enforced by the
        # non-overlapping boundary definitions above (onset_end == ictal_start,
        # offset_start == ictal_end).
        label_regions.append({
            'ictal_start':  ictal_start,    # sample index: seizure start
            'ictal_end':    ictal_end,      # sample index: seizure end (exclusive)
            'onset_start':  onset_start,    # sample index: onset region start
            'onset_end':    onset_end,      # sample index: onset region end (exclusive)
            'offset_start': offset_start,   # sample index: offset region start
            'offset_end':   offset_end      # sample index: offset region end (exclusive)
        })

    return label_regions   # return list of per-seizure region dicts with integer sample indices


In [ ]:
# Cell 10: segment_and_label() — sliding-window segmentation of labelled regions

def segment_and_label(
    signal,               # 1-D float32 NumPy array returned by clean_and_normalise_signal()
    label_regions,        # list of region dicts returned by compute_segment_labels()
    sampling_rate=500,    # samples per second — must match the rate used during preprocessing
    window_sec=5,         # duration of each segment in seconds
    overlap_sec=2.5       # overlap between consecutive windows in seconds
):
    """
    Slide a fixed-length window over each labelled region and collect segments.

    Non-seizure (nonictal) segments are derived automatically as all parts of the
    signal not covered by any ictal, onset, or offset region.

    Parameters
    ----------
    signal        : np.ndarray    Preprocessed 1-D EEG signal, shape (n_samples,).
    label_regions : list of dict  Region dicts from compute_segment_labels().
    sampling_rate : int           Samples per second (default 500).
    window_sec    : float         Window length in seconds (default 5 -> 2500 samples).
    overlap_sec   : float         Overlap in seconds (default 2.5 -> step of 1250 samples).

    Returns
    -------
    ictal_segments    : list of np.ndarray   Each array has shape (window_samples,).
    onset_segments    : list of np.ndarray   Each array has shape (window_samples,).
    offset_segments   : list of np.ndarray   Each array has shape (window_samples,).
    nonictal_segments : list of np.ndarray   Each array has shape (window_samples,).
    """

    window_samples = int(window_sec * sampling_rate)             # number of samples per window (e.g. 5 * 500 = 2500)
    step_samples   = int((window_sec - overlap_sec) * sampling_rate)  # step between windows (e.g. 2.5 * 500 = 1250)

    ictal_segments    = []   # windows falling inside a seizure
    onset_segments    = []   # windows from the 30 s before each seizure
    offset_segments   = []   # windows from the 30 s after each seizure
    nonictal_segments = []   # windows from all remaining non-seizure periods

    def slide(start, end, out_list):
        """Slide a window from start to end, appending only complete windows."""
        pos = start                          # initialise window position at region start
        while pos + window_samples <= end:   # discard any incomplete window at the region boundary
            out_list.append(signal[pos : pos + window_samples].copy())  # copy to avoid referencing full signal
            pos += step_samples              # advance by step size for the next window

    # ── Segment ictal, onset, and offset regions ──────────────────────────────
    for region in label_regions:   # one dict per annotated seizure
        slide(region['ictal_start'],  region['ictal_end'],  ictal_segments)   # seizure samples
        slide(region['onset_start'],  region['onset_end'],  onset_segments)   # pre-seizure buffer
        slide(region['offset_start'], region['offset_end'], offset_segments)  # post-seizure buffer

    # ── Compute nonictal intervals as the complement of all protected regions ──
    # Collect the full protected window for each seizure (onset start -> offset end).
    protected = [[r['onset_start'], r['offset_end']] for r in label_regions]  # one interval per seizure

    protected.sort(key=lambda x: x[0])   # sort intervals by start position so gaps can be found in one pass

    # Merge any overlapping or adjacent protected intervals.
    merged = []                           # list of non-overlapping merged intervals
    for start, end in protected:
        if merged and start <= merged[-1][1]:        # current interval overlaps the previous one
            merged[-1][1] = max(merged[-1][1], end)  # extend the previous interval to cover both
        else:
            merged.append([start, end])              # no overlap — add as a new interval

    # The nonictal intervals are the gaps between (and outside) the merged protected intervals.
    nonictal_intervals = []   # list of (start, end) sample index pairs for non-seizure regions
    prev_end = 0              # track where the last protected interval ended; start from signal beginning

    for start, end in merged:
        if prev_end < start:                              # there is a gap before this protected interval
            nonictal_intervals.append((prev_end, start))  # that gap is a nonictal region
        prev_end = end                                    # advance past the current protected interval

    if prev_end < len(signal):                            # check for a nonictal region after the last seizure
        nonictal_intervals.append((prev_end, len(signal)))  # tail of the recording is nonictal

    # Slide windows over all nonictal intervals.
    for start, end in nonictal_intervals:
        slide(start, end, nonictal_segments)   # segment each nonictal region the same way as labelled regions

    return ictal_segments, onset_segments, offset_segments, nonictal_segments  # return all four class lists

In [ ]:
# Cell 11: save_labelled_segments() — save each segment to a per-mouse sub-subfolder

def save_labelled_segments(
    ictal_segments,     # list of 1-D NumPy arrays for the ictal class
    onset_segments,     # list of 1-D NumPy arrays for the onset class
    offset_segments,    # list of 1-D NumPy arrays for the offset class
    nonictal_segments,  # list of 1-D NumPy arrays for the nonictal (non-seizure) class
    output_dir,         # root output directory; class subfolders are created inside it
    mouse_id            # subject identifier extracted from the EDF filename (e.g. 'm1')
):
    """
    Save labelled segments to a two-level folder structure:
        output_dir/{label}/{mouse_id}/{mouse_id}_{label}_{index:05d}.npy

    Creating a per-mouse sub-subfolder inside each class folder means that running
    the pipeline on additional mice never conflicts with existing files, and all
    segments for one mouse are grouped together under their ID.

    Parameters
    ----------
    ictal_segments    : list of np.ndarray   Ictal windows.
    onset_segments    : list of np.ndarray   Onset windows.
    offset_segments   : list of np.ndarray   Offset windows.
    nonictal_segments : list of np.ndarray   Nonictal (non-seizure) windows.
    output_dir        : str                  Root output directory.
    mouse_id          : str                  Subject identifier (e.g. 'm1').
    """

    # Map each class label to its corresponding list of segments.
    class_map = {
        'ictal':    ictal_segments,     # seizure windows -> ictal/{mouse_id}/
        'onset':    onset_segments,     # pre-seizure windows -> onset/{mouse_id}/
        'offset':   offset_segments,    # post-seizure windows -> offset/{mouse_id}/
        'nonictal': nonictal_segments   # non-seizure windows -> nonictal/{mouse_id}/
    }

    for label, segments in class_map.items():   # iterate over each of the four classes

        # Build the two-level path: output_dir / label / mouse_id
        class_dir = os.path.join(output_dir, label)            # first level:  e.g. C:/data/segments/ictal/
        mouse_dir = os.path.join(class_dir, mouse_id)          # second level: e.g. C:/data/segments/ictal/m1/

        os.makedirs(mouse_dir, exist_ok=True)   # create both levels at once; exist_ok avoids errors on re-runs

        for idx, segment in enumerate(segments):   # iterate over every segment in this class

            filename = f'{mouse_id}_{label}_{idx:05d}.npy'   # e.g. m1_ictal_00012.npy
            filepath = os.path.join(mouse_dir, filename)      # full path: .../ictal/m1/m1_ictal_00012.npy

            np.save(filepath, segment)   # save the 1-D NumPy array to disk in NumPy binary format

        # Confirm the output location and segment count for this class after all files are written.
        print(f'  [{label:>8s}]  {len(segments):5d} segments  ->  {mouse_dir}')


In [ ]:
# Cell 12: run_preprocessing_pipeline() — orchestrate all steps for one recording

def run_preprocessing_pipeline(
    edf_path,             # full path to the .edf recording file
    annotation_path,      # full path to the matching .xlsx annotation file
    output_dir,           # root directory where labelled .npy segments will be saved
    channel_name,         # EEG channel label to extract from the EDF file
    sampling_rate=500     # expected sampling rate of the recording in Hz
):
    """
    Run the complete preprocessing and segmentation pipeline for one mouse recording.

    Steps
    -----
    1. Extract mouse_id from the EDF filename.
    2. Load and preprocess the EEG signal with clean_and_normalise_signal().
    3. Load the seizure annotations with load_annotations().
    4. Extract the recording start time with get_recording_start_time().
    5. Compute sample-level label regions with compute_segment_labels().
    6. Generate labelled windows with segment_and_label().
    7. Save segments to disk with save_labelled_segments().
    8. Print a summary of saved segment counts per class.

    Parameters
    ----------
    edf_path        : str   Full path to the .edf file.
    annotation_path : str   Full path to the .xlsx annotation file.
    output_dir      : str   Root output directory for labelled segments.
    channel_name    : str   EEG channel label to extract.
    sampling_rate   : int   Sampling rate in Hz (default 500).
    """

    # ── Step 1: Extract mouse ID from the EDF filename ────────────────────────
    mouse_id = os.path.splitext(os.path.basename(edf_path))[0]   # e.g. '/data/m1.edf' -> 'm1'
    print(f'Processing mouse: {mouse_id}')                        # confirm which subject is being processed

    # ── Step 2: Load and preprocess the EEG signal ────────────────────────────
    signal = clean_and_normalise_signal(   # returns a clean 1-D float32 NumPy array
        edf_path=edf_path,                 # full path to the EDF file
        channel_name=channel_name,         # which EEG channel to extract
        verbose=False                      # suppress step-by-step output
    )
    print(f'  Signal loaded   : {signal.shape[0]} samples')   # report total number of samples

    # ── Step 3: Load seizure annotations ──────────────────────────────────────
    annotations_df = load_annotations(annotation_path)   # DataFrame with start_time and end_time columns
    print(f'  Annotations     : {len(annotations_df)} seizure(s) found')

    # ── Step 4: Extract recording start time from EDF header ──────────────────
    recording_start = get_recording_start_time(edf_path)   # timezone-naive datetime from EDF header

    # ── Step 5: Convert annotation datetimes to sample-level region boundaries
    label_regions = compute_segment_labels(   # list of dicts with ictal/onset/offset sample indices
        annotations_df=annotations_df,
        recording_start=recording_start,
        sampling_rate=sampling_rate
    )

    # ── Step 6: Slide windows across all labelled regions ─────────────────────
    ictal_segs, onset_segs, offset_segs, nonictal_segs = segment_and_label(  # unpack all four class lists
        signal=signal,               # preprocessed 1-D signal
        label_regions=label_regions, # sample-index region boundaries
        sampling_rate=sampling_rate  # sampling rate in Hz
    )

    # ── Step 7: Save each segment as an individual .npy file ──────────────────
    save_labelled_segments(             # writes files into ictal/, onset/, offset/, nonictal/ subfolders
        ictal_segments=ictal_segs,      # seizure windows
        onset_segments=onset_segs,      # pre-seizure windows
        offset_segments=offset_segs,    # post-seizure windows
        nonictal_segments=nonictal_segs, # non-seizure windows
        output_dir=output_dir,          # root output directory
        mouse_id=mouse_id               # subject identifier for filenames
    )

    # ── Step 8: Print summary ─────────────────────────────────────────────────
    print(f'  Segments saved  : {len(ictal_segs)} ictal  |  {len(onset_segs)} onset  |  '
          f'{len(offset_segs)} offset  |  {len(nonictal_segs)} nonictal')
    print(f'  Output          : {output_dir}')   # confirm where files were written

In [ ]:
# Cell 13: Run the full pipeline for every mouse EEG recording in a folder

data_dir        = r'C:/data/edfs'        # folder containing all EDF files (one per mouse)
annotation_dir  = r'C:/data/annotations' # folder containing matching .xlsx files (one per mouse)
output_dir      = r'C:/data/segments'    # root output folder; ictal/, onset/, offset/ created inside
channel_name    = 'EEG Fp1'             # EEG channel label to extract — must be the same in every file
sampling_rate   = 500                    # sampling rate in Hz — must match the actual recordings

# Collect all EDF files in the data folder, sorted for reproducible order
edf_files = sorted(
    f for f in os.listdir(data_dir)      # list every entry in the data folder
    if f.lower().endswith('.edf')        # keep only .edf files
)

print(f'Found {len(edf_files)} EDF file(s) to process.')  # report how many mice will be processed

failed = []   # track any files that raise an error so the loop does not stop on a bad file

for edf_filename in edf_files:   # iterate over every mouse recording

    edf_path        = os.path.join(data_dir, edf_filename)                          # full path to the EDF file
    mouse_id        = os.path.splitext(edf_filename)[0]                             # e.g. 'm1.edf' -> 'm1'
    annotation_path = os.path.join(annotation_dir, mouse_id + '.xlsx')              # matching annotation file e.g. m1.xlsx

    if not os.path.exists(annotation_path):                                          # skip if no annotation file exists for this mouse
        print(f'[SKIP] No annotation file found for {mouse_id}: {annotation_path}')
        failed.append(mouse_id)
        continue

    try:
        run_preprocessing_pipeline(      # run all steps for this mouse: preprocess -> segment -> save
            edf_path=edf_path,           # full path to the EDF recording
            annotation_path=annotation_path,  # full path to the matching Excel annotation file
            output_dir=output_dir,       # root directory where labelled segments will be saved
            channel_name=channel_name,   # EEG channel to extract
            sampling_rate=sampling_rate  # sampling rate in Hz
        )

    except Exception as exc:                               # catch unexpected errors so the loop continues
        print(f'[ERROR] {mouse_id} failed: {exc}')        # report the error without stopping the batch
        failed.append(mouse_id)                            # record this mouse as failed

# Final summary
print(f'\nDone. {len(edf_files) - len(failed)}/{len(edf_files)} mice processed successfully.')
if failed:
    print(f'Failed: {failed}')   # list any mice that were skipped or raised errors